In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("dirty_cafe_sales.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [2]:
# Check dataset information
print("Dataset Information:")
df.info()

print("\nMissing Values in Each Column:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nDataset Shape:")
print(df.shape)

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB

Missing Values in Each Column:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Duplicate Rows:
0

Dataset Shape:
(10000, 8)


In [3]:
print("Missing Values in Each Column:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nDataset Shape:")
print(df.shape)

Missing Values in Each Column:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Duplicate Rows:
0

Dataset Shape:
(10000, 8)


In [4]:
# Calculate missing values percentage
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100

missing_report = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percent.round(2)
})

print(missing_report)

                  Missing Count  Missing Percentage
Transaction ID                0                0.00
Item                        333                3.33
Quantity                    138                1.38
Price Per Unit              179                1.79
Total Spent                 173                1.73
Payment Method             2579               25.79
Location                   3265               32.65
Transaction Date            159                1.59


In [5]:
# Check ERROR and UNKNOWN values
for col in df.columns:
    print(f"\n{col}:")
    print(df[col].astype(str).str.strip().isin(
        ["ERROR", "UNKNOWN"]
    ).sum())


Transaction ID:
0

Item:
636

Quantity:
341

Price Per Unit:
354

Total Spent:
329

Payment Method:
599

Location:
696

Transaction Date:
301


In [6]:
# Create a copy of the original dataset
cleaned_df = df.copy()

# Replace invalid values with missing values
cleaned_df = cleaned_df.replace(
    ["ERROR", "UNKNOWN"], np.nan
)

print("Invalid values replaced successfully!")

print("\nMissing values after replacement:")
print(cleaned_df.isnull().sum())


Invalid values replaced successfully!

Missing values after replacement:
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


In [7]:
# Fill missing values using suitable methods

# Categorical columns: fill with "Unknown"
categorical_cols = ["Item", "Payment Method", "Location"]

for col in categorical_cols:
    cleaned_df[col] = cleaned_df[col].fillna("Unknown")

# Numerical columns: fill with median
numerical_cols = ["Quantity", "Price Per Unit", "Total Spent"]

for col in numerical_cols:
    cleaned_df[col] = pd.to_numeric(
        cleaned_df[col], errors="coerce"
    )
    cleaned_df[col] = cleaned_df[col].fillna(
        cleaned_df[col].median()
    )

# Transaction Date: convert to date format
cleaned_df["Transaction Date"] = pd.to_datetime(
    cleaned_df["Transaction Date"], errors="coerce"
)

print("Missing values handled!")
print("\nRemaining missing values:")
print(cleaned_df.isnull().sum())

Missing values handled!

Remaining missing values:
Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    460
dtype: int64


In [8]:
# Handle missing transaction dates
cleaned_df["Transaction Date"] = cleaned_df["Transaction Date"].fillna(
    cleaned_df["Transaction Date"].mode()[0]
)

# Remove duplicate rows
duplicates_before = cleaned_df.duplicated().sum()
cleaned_df = cleaned_df.drop_duplicates()

print("Duplicates removed:", duplicates_before)
print("Remaining missing values:")
print(cleaned_df.isnull().sum())

print("\nFinal dataset shape:", cleaned_df.shape)

Duplicates removed: 0
Remaining missing values:
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

Final dataset shape: (10000, 8)


In [9]:
# Detect outliers using the IQR method

numerical_cols = ["Quantity", "Price Per Unit", "Total Spent"]

for col in numerical_cols:
    Q1 = cleaned_df[col].quantile(0.25)
    Q3 = cleaned_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = cleaned_df[
        (cleaned_df[col] < lower) |
        (cleaned_df[col] > upper)
    ]

    print(f"\n{col}")
    print("Lower limit:", lower)
    print("Upper limit:", upper)
    print("Outlier count:", len(outliers))


Quantity
Lower limit: -1.0
Upper limit: 7.0
Outlier count: 0

Price Per Unit
Lower limit: -1.0
Upper limit: 7.0
Outlier count: 0

Total Spent
Lower limit: -8.0
Upper limit: 24.0
Outlier count: 259


In [10]:
# Correct data types
cleaned_df["Transaction ID"] = cleaned_df["Transaction ID"].astype("string")
cleaned_df["Item"] = cleaned_df["Item"].astype("string")
cleaned_df["Payment Method"] = cleaned_df["Payment Method"].astype("string")
cleaned_df["Location"] = cleaned_df["Location"].astype("string")

cleaned_df["Quantity"] = cleaned_df["Quantity"].astype("Int64")
cleaned_df["Price Per Unit"] = cleaned_df["Price Per Unit"].astype(float)
cleaned_df["Total Spent"] = cleaned_df["Total Spent"].astype(float)

print("Updated Data Types:")
print(cleaned_df.dtypes)

Updated Data Types:
Transaction ID              string
Item                        string
Quantity                     Int64
Price Per Unit             float64
Total Spent                float64
Payment Method              string
Location                    string
Transaction Date    datetime64[us]
dtype: object


In [11]:
# Save cleaned dataset
cleaned_df.to_csv("Keerthana_Task3_Cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")
print("Final dataset shape:", cleaned_df.shape)
print("Remaining missing values:", cleaned_df.isnull().sum().sum())

Cleaned dataset saved successfully!
Final dataset shape: (10000, 8)
Remaining missing values: 0


In [12]:
# Before and after cleaning comparison

summary = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Missing Values",
        "Duplicate Rows"
    ],
    "Before Cleaning": [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum()
    ],
    "After Cleaning": [
        cleaned_df.shape[0],
        cleaned_df.shape[1],
        cleaned_df.isnull().sum().sum(),
        cleaned_df.duplicated().sum()
    ]
})

print(summary.to_string(index=False))

        Metric  Before Cleaning  After Cleaning
    Total Rows            10000           10000
 Total Columns                8               8
Missing Values             6826               0
Duplicate Rows                0               0


In [13]:
# Final data quality verification

print("Dataset Shape:", cleaned_df.shape)

print("\nMissing Values:")
print(cleaned_df.isnull().sum())

print("\nDuplicate Rows:", cleaned_df.duplicated().sum())

print("\nRemaining ERROR / UNKNOWN values:")
for col in cleaned_df.columns:
    count = cleaned_df[col].astype(str).str.strip().isin(
        ["ERROR", "UNKNOWN"]
    ).sum()
    if count > 0:
        print(col, ":", count)

print("\nFinal verification completed!")

Dataset Shape: (10000, 8)

Missing Values:
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

Duplicate Rows: 0

Remaining ERROR / UNKNOWN values:

Final verification completed!
